In [5]:
#membuat dataset terlebih dahulu
import os
if not os.path.exists("data_transaksi_ecommer.csv"):
    import numpy as np
    import pandas as pd
    np.random.seed(42)
    n = 600
    kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
    kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
    metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
    tanggal_range = pd.date_range("2026-07-01", "2026-07-31", freq="D")
    data = {
        "order_id": [f"ORD-{1000+i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25,0.25,0.20,0.15,0.15]),
        "kota": np.random.choice(kota_list, size=n),
        "unit_terjual": np.random.randint(1, 10, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000, 500000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n, p=[0.35, 0.30, 0.20, 0.15]),
    }
    pd.DataFrame(data).to_csv("data_transaksi_ecommerce.csv", index=False)
    print("Dataset dibuat ulang.")
else:
    print("Dataset sudah tersedia.")

Dataset dibuat ulang.


In [10]:
#menjalankan spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Latihan4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
print("Siap. Jumlah baris: ", df.count())

Siap. Jumlah baris:  600


In [14]:
# tampilkan kolom order_id, kategori, dan total_pendapatan untuk transaksi dengan metode_pembayaran bernilai E-wallet
df.filter(col("metode_pembayaran") == "E-Wallet")\
    .select("order_id", "kategori", "total_pendapatan")\
    .show(5)

+--------+--------------------+----------------+
|order_id|            kategori|total_pendapatan|
+--------+--------------------+----------------+
|ORD-1000|Kesehatan & Kecan...|         2250000|
|ORD-1003|          Elektronik|          150000|
|ORD-1006|   Makanan & Minuman|           25000|
|ORD-1013|   Makanan & Minuman|          900000|
|ORD-1014|             Fashion|           75000|
+--------+--------------------+----------------+
only showing top 5 rows



In [16]:
# hitung total pendapatan per kategori (bukan per kota), urutkan dari yang tertinggi
from pyspark.sql.functions import sum as spark_sum, col

df.groupBy("kategori")\
  .agg(spark_sum("total_pendapatan").alias("total_pendapatan_kategori"))\
  .orderBy(col("total_pendapatan_kategori").desc())\
  .show()

+--------------------+-------------------------+
|            kategori|total_pendapatan_kategori|
+--------------------+-------------------------+
|             Fashion|                124825000|
|          Elektronik|                110600000|
|   Makanan & Minuman|                 95400000|
|Kesehatan & Kecan...|                 82200000|
|        Rumah Tangga|                 77350000|
+--------------------+-------------------------+



In [17]:
# tampilkan jumlah transaksi untuk asing-masing metode_pembayaran (tidak memakai agg)
df.groupBy("metode_pembayaran").count().show()

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|              COD|  114|
|    Transfer Bank|  208|
|     Kartu Kredit|   85|
|         E-Wallet|  193|
+-----------------+-----+



In [18]:
# pilih select apabila perintah meminta untuk menampilkan kolom
# pilih filter apabila perintah meminta menampilkan baris
# pilih groupBy dan agg untuk meringkas kelompok

In [21]:
!hdfs dfs -mkdir -p /user/azka
!hdfs dfs -put -f data_transaksi_ecommerce.csv /user/azka/

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Latihan").getOrCreate()

# membaca HDFS
df = spark.read.csv("hdfs://localhost:9000/user/azka/data_transaksi_ecommerce.csv", header=True, inferSchema=True)

df.printSchema()
print("Jumlah baris: ", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)

Jumlah baris:  600
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|ORD-1000|2026-07-07 00:00:00|Kesehatan & Kecan...|  Magelang|           9|      250000|         E-Wallet|
|ORD-1001|2026-07-20 00:00:00|   Makanan & Minuman|Yogyakarta|           9|       50000|     Kartu Kredit|
|ORD-1002|2026-07-29 00:00:00|             Fashion| Purworejo|           2|       50000|     Kartu Kredit|
|ORD-1003|2026-07-15 00:00:00|      

In [25]:
# Kolom rating memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan df.na.fill() atau df.na.drop()
from pyspark.sql.functions import col

# 1. Tampilkan jumlah nilai kosong
jumlah_null = df.filter(col("rating").isNull()).count()
print("Jumlah rating kosong:", jumlah_null)

# 2. Isi nilai kosong dengan 0
df_fill = df.na.fill({"rating": 0})

# 3. Verifikasi
print("Jumlah baris sebelum:", df.count())
print("Jumlah baris sesudah fill:", df_fill.count())   
print("Sisa null:", df_fill.filter(col("rating").isNull()).count())

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `rating` cannot be resolved. Did you mean one of the following? [`kota`, `tanggal`, `kategori`, `order_id`, `harga_satuan`].;
'Filter isnull('rating)
+- Relation [order_id#253,tanggal#254,kategori#255,kota#256,unit_terjual#257,harga_satuan#258,metode_pembayaran#259] csv
